In [140]:
import os
import json
import io
import urllib.parse as up

import re
from datetime import datetime
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload

import numpy as np
import pandas as pd

# Connect to Google Drive
import gspread
import gspread_dataframe
from google.oauth2.service_account import Credentials
from google.oauth2 import service_account
from gspread_dataframe import set_with_dataframe
from gspread_dataframe import get_as_dataframe





In [141]:


# SILENCIADO CON # + ESPACIO

# 1. Fetch credentials from environment variable
# creds_env = os.environ.get("GDRIVE_CREDENTIALS_KC")

# if not creds_env:
#     raise ValueError("Environment variable 'GDRIVE_CREDENTIALS' was not found.")

# creds_json = json.loads(creds_env)

# 2. Define required scopes
# scopes = [
#     "https://www.googleapis.com/auth/spreadsheets",
#     "https://www.googleapis.com/auth/drive",
# ]

# 3. Authenticate service account
# creds = service_account.Credentials.from_service_account_info(
#     creds_json, scopes=scopes
# )

# 4. Initialize Google API clients
# drive_service = build("drive", "v3", credentials=creds)
# sheets_service = build("sheets", "v4", credentials=creds)

# gc = gspread.authorize(creds)

# print("Google Drive and Sheets services successfully initialized.")

In [142]:

from google.colab import drive
# Connect to Google Drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [143]:
# Connect google API to work with files in Drive

json_key_path = '/content/drive/Shareddrives/kimberly clark/Berni_Sprinklr_Data/dashboard-kc-498614-c17a185c7dd0.json'
# Set up the Google API scopes
scopes = [
    'https://www.googleapis.com/auth/spreadsheets',
    'https://www.googleapis.com/auth/drive'
]

# Authenticate using the JSON file from your Drive
creds = Credentials.from_service_account_file(json_key_path, scopes=scopes)
gc = gspread.authorize(creds)

print("Successfully authenticated using Drive-stored JSON key!")

Successfully authenticated using Drive-stored JSON key!


In [144]:
# ID of the Drive folder that holds the Sprinklr Paid exports.
# Get it from the folder URL: drive.google.com/drive/folders/<THIS_IS_THE_ID>
DRIVE_FOLDER_ID = "1k8NDz3qxQ9ffZzkS2EsWT1tNSk3ghklU"

# File name pattern: DDMMYYYY.xlsx (e.g. 31072026.xlsx)
FILENAME_PATTERN = re.compile(r"^(\d{2})(\d{2})(\d{4})\.xlsx$")

In [145]:
# Build the Drive service reusing the service-account creds already created above
drive_service = build("drive", "v3", credentials=creds)

def find_latest_file(drive_service):
    query = f"'{DRIVE_FOLDER_ID}' in parents and trashed = false"
    results = drive_service.files().list(
        q=query,
        fields="files(id, name)",
        pageSize=1000,
        supportsAllDrives=True,
        includeItemsFromAllDrives=True,
        corpora="allDrives",
    ).execute()

    files = results.get("files", [])
    print(f"[DEBUG] Archivos visibles en la carpeta {DRIVE_FOLDER_ID}: {len(files)}")
    for f in files:
        print(f"[DEBUG]  - {f['name']} (id={f['id']})")

    candidates = []
    for f in files:
        m = FILENAME_PATTERN.match(f["name"])
        if not m:
            continue
        dd, mm, yyyy = m.groups()
        try:
            file_date = datetime(int(yyyy), int(mm), int(dd))
        except ValueError:
            continue
        candidates.append((file_date, f))

    if not files:
        raise RuntimeError(
            "No se retornó ningún archivo para esa carpeta. Probablemente la service "
            "account no tiene acceso, o el ID de la carpeta está mal."
        )
    if not candidates:
        raise RuntimeError(
            "No se encontró ningún archivo con el patrón DDMMYYYY.xlsx en la carpeta. "
            "Revisá la lista [DEBUG] de arriba para ver los nombres reales."
        )

    candidates.sort(key=lambda x: x[0])
    latest_date, latest_file = candidates[-1]
    print(f"Archivo más reciente encontrado: {latest_file['name']} (fecha {latest_date.date()})")
    return latest_file["id"], latest_file["name"]

def download_file(drive_service, file_id, local_path):
    request = drive_service.files().get_media(fileId=file_id, supportsAllDrives=True)
    fh = io.BytesIO()
    downloader = MediaIoBaseDownload(fh, request)
    done = False
    while not done:
        status, done = downloader.next_chunk()
    with open(local_path, "wb") as f:
        f.write(fh.getvalue())
    print(f"Archivo descargado en: {local_path}")

# Find the latest file, download it, and open it
file_id, file_name = find_latest_file(drive_service)
file_path = f"/tmp/{file_name}"
download_file(drive_service, file_id, file_path)

sprinklr_paid_new = pd.read_excel(file_path, sheet_name='WinClap_Paid Media', header=2)
sprinklr_paid_new = sprinklr_paid_new.fillna(0)

[DEBUG] Archivos visibles en la carpeta 1k8NDz3qxQ9ffZzkS2EsWT1tNSk3ghklU: 53
[DEBUG]  - backfill_angie_6m_aug2026.xlsx (id=1_GrKVOnkwHGn0IFg5mvqrEHNKUz6HL7t)
[DEBUG]  - 21082026.xlsx (id=1NN9D1_IA36EpRPYW4TtEWbGdrZfsP-Ox)
[DEBUG]  - 20082026.xlsx (id=1hyhhUwunFcCso6LYDFdXavcfhDYGwDej)
[DEBUG]  - 19082026.xlsx (id=1XkLDsBCLijlQhmxTGNui3N7bx2BjE5u5)
[DEBUG]  - 18082026.xlsx (id=1jTGu6jpxoIsfwVL-FTDosZJ5MYhiB09t)
[DEBUG]  - 14082026.xlsx (id=1z4EOBQou0a_Bop-V0NEBjJ4isMq_apxY)
[DEBUG]  - 13082026.xlsx (id=1YUNwn2LqiuRFKhthtrg-9c3F1RQHlE3q)
[DEBUG]  - COPIA backfill 2026-08-05.xlsx (id=1JmmmAoetV3aHLCjqLsqqfpXq3ibFCDx6)
[DEBUG]  - 30062026.xlsx (id=14Cr7qzkMM7w1pOhDI8xBDrO97ERxXW8L)
[DEBUG]  - 12082026.xlsx (id=1gUHp3rNg-4IaxCyeenDo9Q0D-QsvfGEK)
[DEBUG]  - 11082026.xlsx (id=1XVilSR5nQd7f69nthIJJkgatFKqihZhi)
[DEBUG]  - 10082026.xlsx (id=1ocNES4PSzeU5G13kttHj4BX8KZekEuFI)
[DEBUG]  - 09082026.xlsx (id=1Ol3FpVVzbLPdKKip11BcrK6Q1xhD3sBP)
[DEBUG]  - 07082026.xlsx (id=1k3gmsWGXC9S9Idd07m6fwGwnuq

In [146]:
# Extract organic id from 'Ad Variant Name'
def extract_id(url):
    if pd.isna(url):
        return None
    u = str(url).strip().split("?")[0].rstrip("/")   # drop query string & trailing slash

    # TikTok: .../video/<id>  or  .../photo/<id>   (numeric id)
    m = re.search(r"/(?:video|photo)/(\d+)", u)
    if m:
        return m.group(1)

    # Instagram: /reel/<code>, /p/<code>, /tv/<code>   (alphanumeric shortcode)
    m = re.search(r"/(?:reel|reels|p|tv)/([A-Za-z0-9_-]+)", u)
    if m:
        return m.group(1)

    return None  # unrecognized pattern

sprinklr_paid_new["Organic_ID"] = sprinklr_paid_new["Ad Post Permalink"].apply(extract_id)

In [147]:
# Keep only the rows that match with organic posts and influencer posts (inner join with the final organic table) -> we do this so the paid data file doesn't become huge due to unnecesary data (ads that are not organic boosting)

# Open organic posts file
organic_posts = gc.open_by_key('1FnauIqLuTe1c2N8Z-HQPy8wambQzBhpbLJY24JMCMNY')
organic_posts = organic_posts.worksheet('Hoja 1')
organic_posts = get_as_dataframe(organic_posts)
organic_posts = organic_posts[['Organic_ID', 'Outbound Message Category', 'Brand (Account)', 'Country of Origin (Account)', 'Published Date', 'Video Views (SUM)']]
organic_posts['Published Date'] = pd.to_datetime(
    organic_posts['Published Date'], errors='coerce'
)
organic_posts['Outbound Message Category'] = (
    organic_posts['Video Views (SUM)'].eq(0).map({True: 'Static', False: 'Video'})
)
organic_posts = organic_posts.drop(columns="Video Views (SUM)", errors="ignore")

# Open influencers posts file
influencer_posts = gc.open_by_key('1QwqDvUu5SAt6PHKBZWWOATkqzE58pN_XDnzo6LMiYOI')
influencer_posts = influencer_posts.worksheet('Sheet1')
influencer_posts = get_as_dataframe(influencer_posts)
influencer_posts = influencer_posts[['Organic_ID', 'format', 'brand', 'Country', 'date_published']]
influencer_posts = influencer_posts[['Organic_ID','format', 'brand', 'Country', 'date_published']].rename(columns={
    'Organic_ID': 'Organic_ID',
    'format': 'Outbound Message Category',
    'brand': 'Brand (Account)',
    'Country': 'Country of Origin (Account)',
    'date_published': 'Published Date'
})
influencer_posts['Published Date'] = pd.to_datetime(
    influencer_posts['Published Date'], errors='coerce'
)
influencer_posts['Outbound Message Category'] = (
    influencer_posts['Outbound Message Category'].eq('Carousel').map({True: 'Static', False: 'Video'})
)

## AGREGAR TODAS LAS COLUMNAS QUE HACEN FALTA ANTES DEL APPEND, DESPUÉS APPENDEAR (CAMBIAR NOMBRE ANTES SI HACE FALTA)




In [148]:
# Append organic and influencers
posts_filter_list = pd.concat([organic_posts, influencer_posts], ignore_index=True)
posts_filter_list = posts_filter_list.drop_duplicates(subset="Organic_ID")

In [149]:
# Keep only the rows that match with organic posts and influencer posts (inner join with posts_filter_list) -> we do this so the paid data file doesn't become huge due to unnecesary data (ads that are not organic boosting)

# Filter the rows in sprinklr_paid_new that match with Organic_ID in organic posts
sprinklr_paid_new = sprinklr_paid_new[sprinklr_paid_new['Organic_ID'].isin(posts_filter_list['Organic_ID'])].reset_index(drop=True)
sprinklr_paid_new = sprinklr_paid_new.dropna(subset=['Organic_ID'])
sprinklr_paid_new = sprinklr_paid_new.merge(
    posts_filter_list,
    on="Organic_ID",
    how="left"
)

overlap = sprinklr_paid_new.columns.intersection(posts_filter_list.columns).drop("Organic_ID")
sprinklr_paid_new = sprinklr_paid_new.merge(
    posts_filter_list.drop(columns=overlap),
    on="Organic_ID",
    how="left"
)

sprinklr_paid_new

,Date,Ad Account,Social Network,Paid Initiative Name,Ad Variant Name,Ad Variant Id,Ad Variant,Title,Body,Image URL,...,Negative Sentiment Count (Paid + Organic) (AVG),Facebook Avg. Duration of Video Played (SUM),Ad Post Id,Ad Post Permalink,TikTok Clicks (Destination) (SUM),Organic_ID,Outbound Message Category,Brand (Account),Country of Origin (Account),Published Date
0,2026-07-23,KOTEX_CO_ES_KC-FEM-OMD,TikTok,EM_CO_FemCare_Kotex_2026Q2_Cassandra_Soc_TikTo...,06900199_Other_Me-pasas-la-toalla_NA_InFeed_NO...,TIKTOK_1867741443381282,06900199_Other_Me-pasas-la-toalla_NA_InFeed_NO...,0,Le pedí a mi novio una toalla… claramente no e...,https://prod.cdata.app.sprinklr.com/PAID/312/a...,...,4,0,20284361504,https://www.tiktok.com/@kotexcol/video/7623953...,0,7623953193705229589,Video,Kotex,Colombia,2026-04-01 19:32:21
1,2026-07-23,KOTEX_CO_ES_KC-FEM-OMD,TikTok,EM_CO_FemCare_Kotex_2026Q3_Cassandra_Soc_TikTo...,07099470_Other_MePasasLaToalla_Frmtodo_InFeed_...,TIKTOK_1869540400398642,07099470_Other_MePasasLaToalla_Frmtodo_InFeed_...,0,Le pedí a mi novio una toalla… claramente no e...,https://prod.cdata.app.sprinklr.com/PAID/312/a...,...,4,0,20284361504,https://www.tiktok.com/@kotexcol/video/7623953...,545,7623953193705229589,Video,Kotex,Colombia,2026-04-01 19:32:21
2,2026-07-23,KOTEX_CO_ES_KC-FEM-OMD,TikTok,EM_CO_FemCare_Kotex_2026Q3_Cassandra_Soc_TikTo...,07099470_Other_MePasasLaToalla_Frmtodo_InFeed_...,TIKTOK_1869540400400802,07099470_Other_MePasasLaToalla_Frmtodo_InFeed_...,0,Le pedí a mi novio una toalla… claramente no e...,https://prod.cdata.app.sprinklr.com/PAID/312/a...,...,4,0,20284361504,https://www.tiktok.com/@kotexcol/video/7623953...,222,7623953193705229589,Video,Kotex,Colombia,2026-04-01 19:32:21
3,2026-07-23,KOTEX_CO_ES_KC-FEM-OMD,TikTok,EM_CO_FemCare_Kotex_2026Q3_Cassandra_Soc_TikTo...,07099470_Other_MePasasLaToalla_Frmtodo_InFeed_...,TIKTOK_1869540400400834,07099470_Other_MePasasLaToalla_Frmtodo_InFeed_...,0,Le pedí a mi novio una toalla… claramente no e...,https://prod.cdata.app.sprinklr.com/PAID/312/a...,...,4,0,20284361504,https://www.tiktok.com/@kotexcol/video/7623953...,134,7623953193705229589,Video,Kotex,Colombia,2026-04-01 19:32:21
4,2026-07-23,KOTEX_AR_ES_KC-FEM_OMD,TikTok,EM_AR_FemCare_Kotex_2026Q2_Overnight_Soc_TikTo...,06803148_Other_POV4-1867005549744177_NA_InFeed...,TIKTOK_1867005549745393,06803148_Other_POV4-1867005549744177_NA_InFeed...,0,Hay cosas que no se mueven aunque vos no pares...,0,...,6,0,20373209298,https://www.tiktok.com/@kotexargentina/video/7...,0,7634696851559582994,Video,Kotex,Argentina,2026-04-30 18:23:01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3219,2026-08-21,OMD_AR_ES_KC-BCC_HUGGIES,Instagram,EM_AR_BCC_Hugg_2026Q3_BBC-AO_Soc_Meta_Awa_Baby...,07095289_Other_ElCelularQueTodoEscucha_Meli_Mu...,FACEBOOK_120248675192390318,07095289_Other_ElCelularQueTodoEscucha_Meli_Mu...,0,Si el algoritmo ya nos escucha para vendernos ...,https://prod.cdata.app.sprinklr.com/PAID/312/e...,...,0,2,21537937794,https://www.instagram.com/reel/DcBffc1jRwl/,0,DcBffc1jRwl,Video,Huggies,Argentina,2026-08-14 09:09:08
3220,2026-08-21,OMD_SV_ES_KC-BCC_HUGGIES,Instagram,EM_SV_BCC_Hugg_2026Q3_AON_Soc_Meta_Awa_NATURAL...,07099060_Other_CBA_Wart_Reel_NONPER_DEI-NO_NA_ES,FACEBOOK_120249739374130193,07099060_Other_CBA_Wart_Reel_NONPER_DEI-NO_NA_ES,0,A veces las señales son obvias 😅 La piel de mi...,https://prod.cdata.app.sprinklr.com/PAID/312/7...,...,0,1,20408058378,https://www.instagram.com/reel/DX7xrpDjLqh/,0,DX7xrpDjLqh,Video,Huggies,Costa Rica,2026-05-04 17:46:35
3221,2026-08-21,OMD_SV_ES_KC-BCC_HUGGIES,Instagram,EM_SV_BCC_Hugg_2026Q3_AON_Soc_Meta_Awa_NATURAL...,07099060_Other_ugc-soft-confort_Wart_Reel_NONP...,FACEBOOK_120249739602150193,07099060_Other_ugc-soft-confort_Wart_Reel_NONP...,0,"Que baile, salte y explore sin límites 💚 Con l...",https://prod.cdata.app.sprinklr.com/PAID/312/b...,...,2,1,20482380956,https://www.instagram.com/reel/DYQec5xAYiL/,0,DYQec5xAYiL,Video,

In [150]:
# Regroup countries based on KC country grouping
mask1 = (
    sprinklr_paid_new['Country of Origin (Account)'].isin(
        ['Costa Rica', 'El Salvador', 'Guatemala', 'Honduras',
         'Nicaragua', 'Puerto Rico', 'Panama', 'Dominican Republic']
    )
) & (sprinklr_paid_new['Brand (Account)'] == 'Huggies')
sprinklr_paid_new.loc[mask1, 'Country of Origin (Account)'] = 'CAM'

sprinklr_paid_new.loc[
    sprinklr_paid_new['Brand (Account)'] == 'Plenitud',
    'Country of Origin (Account)'
] = 'Latam'

mask2 = (sprinklr_paid_new['Brand (Account)'] == 'Plenitud') & (
    sprinklr_paid_new['Social Network'] == 'TikTok'
)
sprinklr_paid_new.loc[mask2, 'Country of Origin (Account)'] = 'Latam'

In [151]:
# Correct column 'Spent (USD) in USD (SUM)' format
sprinklr_paid_new['Spent (USD) in USD (SUM)'] = (
    sprinklr_paid_new['Spent (USD) in USD (SUM)']
    .str.replace('$', '', regex=False)
    .str.replace(',', '', regex=False)
    .astype(float)
)

In [152]:
# Group values (sum) by Organic Id, Social Network, Date
sprinklr_paid_new_consolidated = sprinklr_paid_new.groupby(['Organic_ID','Date', 'Ad Account', 'Social Network', 'Paid Initiative Name', 'Ad Variant Name', 'Ad Variant Id', 'Ad Variant', 'Title',
    'Body', 'Image URL', 'Ad Post Id', 'Outbound Message Category', 'Brand (Account)', 'Country of Origin (Account)', 'Published Date', 'Ad Post Permalink'], as_index=False)[[
    'Impressions (SUM)',
    'Spent (USD) in USD (SUM)',
    'TikTok Video views (SUM)',
    'TikTok 6-second video views (SUM)',
    'Facebook Video Plays (3 sec) (SUM)',
    'Facebook Video Plays to 25% (SUM)',
    'TikTok Paid comments (SUM)',
    'TikTok Paid shares (SUM)',
    'TikTok Paid likes (SUM)',
    'Facebook Post Comments (SUM)',
    'Facebook Post Shares (SUM)',
    'Facebook Post Likes (SUM)',
    'Facebook Link Clicks (SUM)',
    'Facebook Reactions (SUM)',
    'Facebook Post Saves (SUM)',
    'Neutral Sentiment Count (Paid + Organic) (AVG)',
    'Positive Sentiment Count (Paid + Organic) (AVG)',
    'Negative Sentiment Count (Paid + Organic) (AVG)',
    'Facebook Avg. Duration of Video Played (SUM)',
    'TikTok Clicks (Destination) (SUM)'
]].sum()

sprinklr_paid_new_consolidated

,Organic_ID,Date,Ad Account,Social Network,Paid Initiative Name,Ad Variant Name,Ad Variant Id,Ad Variant,Title,Body,...,Facebook Post Shares (SUM),Facebook Post Likes (SUM),Facebook Link Clicks (SUM),Facebook Reactions (SUM),Facebook Post Saves (SUM),Neutral Sentiment Count (Paid + Organic) (AVG),Positive Sentiment Count (Paid + Organic) (AVG),Negative Sentiment Count (Paid + Organic) (AVG),Facebook Avg. Duration of Video Played (SUM),TikTok Clicks (Destination) (SUM)
0,7606079619032157461,2026-07-31,KOTEX_AR_ES_KC-FEM_OMD,TikTok,EM_AR_FemCare_Kotex_2026Q1_Verano-Sin-Tabu_Soc...,06627095_Other_vestirse-de-blanco_NA_Reel_NONP...,TIKTOK_1857654716946529,06627095_Other_vestirse-de-blanco_NA_Reel_NONP...,0,0,...,0,0,0,0,0,0,0,0,0,0
1,7606079619032157461,2026-08-03,KOTEX_AR_ES_KC-FEM_OMD,TikTok,EM_AR_FemCare_Kotex_2026Q1_Verano-Sin-Tabu_Soc...,06627095_Other_vestirse-de-blanco_NA_Reel_NONP...,TIKTOK_1857654716946529,06627095_Other_vestirse-de-blanco_NA_Reel_NONP...,0,0,...,0,0,0,0,0,0,0,0,0,0
2,7607522403224194325,2026-08-05,KOTEX_AR_ES_KC-FEM_OMD,TikTok,EM_AR_FemCare_Kotex_2026Q1_Verano-Sin-Tabu_Soc...,06627095_Other_lulilucero_NA_Reel_NONPER_DEI-N...,TIKTOK_1857655001244946,06627095_Other_lulilucero_NA_Reel_NONPER_DEI-N...,0,0,...,0,0,0,0,0,0,0,0,0,0
3,7623953193705229589,2026-07-23,KOTEX_CO_ES_KC-FEM-OMD,TikTok,EM_CO_FemCare_Kotex_2026Q2_Cassandra_Soc_TikTo...,06900199_Other_Me-pasas-la-toalla_NA_InFeed_NO...,TIKTOK_1867741443381282,06900199_Other_Me-pasas-la-toalla_NA_InFeed_NO...,0,Le pedí a mi novio una toalla… claramente no e...,...,0,0,0,0,0,5,4,4,0,0
4,7623953193705229589,2026-07-23,KOTEX_CO_ES_KC-FEM-OMD,TikTok,EM_CO_FemCare_Kotex_2026Q3_Cassandra_Soc_TikTo...,07099470_Other_MePasasLaToalla_Frmtodo_InFeed_...,TIKTOK_1869540400398642,07099470_Other_MePasasLaToalla_Frmtodo_InFeed_...,0,Le pedí a mi novio una toalla… claramente no e...,...,0,0,0,0,0,5,4,4,0,545
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3219,DcBffc1jRwl,2026-08-20,OMD_AR_ES_KC-BCC_HUGGIES,Instagram,EM_AR_BCC_Hugg_2026Q3_BBC-AO_Soc_Meta_Awa_Baby...,07095289_Other_ElCelularQueTodoEscucha_Meli_Mu...,FACEBOOK_120248675192390318,07095289_Other_ElCelularQueTodoEscucha_Meli_Mu...,0,Si el algoritmo ya nos escucha para vendernos ...,...,1,17,45,17,0,0,0,0,1,0
3220,DcBffc1jRwl,2026-08-21,OMD_AR_ES_KC-BCC_HUGGIES,Instagram,EM_AR_BCC_Hugg_2026Q3_BBC-AO_Soc_Meta_Awa_Baby...,07095289_Other_ElCelularQueTodoEscucha_Meli_Mu...,FACEBOOK_120248675192390318,07095289_Other_ElCelularQueTodoEscucha_Meli_Mu...,0,Si el algoritmo ya nos escucha para vendernos ...,...,0,1,0,1,0,0,0,0,2,0
3221,DcHfMaPhgKr,2026-08-19,(NUEVA) KOTEX_CR_ES_KC-FEM_OMD_BMCR,Instagram,EM_CR_FemCare_Kotex_2026Q3_AON2026_Soc_Meta_Aw...,07236378_1080x1920_Natural-Care-101_NA_1202494...,FACEBOOK_120249439265750115,07236378_1080x1920_Natural-Care-101_NA_1202494...,0,"Holi 🎀✨ Soy su CM, la que está detrás de panta...",...,0,16,29,16,0,0,0,0,0,0
3222,DcHfMaPhgKr,2026-08-20,(NUEVA) KOTEX_CR_ES_KC-FEM_OMD_BMCR,Instagram,EM_CR_FemCare_Kotex_2026Q3_AON2026_Soc_Meta_Aw...,07236378_1080x1920_Natural-Care-101_NA_1202494...,FACEBOOK_120249439265750115,07236378_1080x1920_Natural-Care-101_NA_1202494...,0,"Holi 🎀✨ Soy su CM, la que está detrás de panta...",...,0,14,36,14,0,0,0,0,0,0


In [153]:
# Update historic paid file with new data
# Open historic paid file
historic_paid = gc.open_by_key('1giT7UA49YozfGI8kDNY-XYe609CA9hg3flzTINgfFjM')
historic_paid = historic_paid.worksheet('Hoja 1')
historic_paid = get_as_dataframe(historic_paid)

# Ensure 'Date' columns share the exact same data type (crucial for exact matching)
sprinklr_paid_new_consolidated['Date'] = pd.to_datetime(sprinklr_paid_new_consolidated['Date'])
historic_paid['Date'] = pd.to_datetime(historic_paid['Date'])

# Append new data and drop duplicates based on the composite key
historic_paid_updated = (
    pd.concat([historic_paid, sprinklr_paid_new_consolidated], ignore_index=True)
    .drop_duplicates(subset=['Date', 'Organic_ID', 'Paid Initiative Name', 'Ad Variant Id'], keep='first')
    .reset_index(drop=True)
)

In [154]:
# Save the consolidated database (historic+new) as the new historic file (replacing previous data with the new consolidation)
# Open the destination sheets file
sh = gc.open_by_key('1giT7UA49YozfGI8kDNY-XYe609CA9hg3flzTINgfFjM')
worksheet = sh.worksheet('Hoja 1')

# Replace old data with new data
set_with_dataframe(worksheet, historic_paid_updated)
print("DataFrame saved successfully!")

DataFrame saved successfully!


In [155]:
# Work with historic file

# Load historic file
sprinklr_paid_historic_file = gc.open_by_key('1giT7UA49YozfGI8kDNY-XYe609CA9hg3flzTINgfFjM')
# Select the specific worksheet (0 is the first tab)
sprinklr_paid_historic_file_worksheet = sprinklr_paid_historic_file.worksheet('Hoja 1')
# Load the data directly into a pandas DataFrame
sprinklr_paid_historic = get_as_dataframe(sprinklr_paid_historic_file_worksheet)

In [156]:
# Work with historic file

# Create metrics
sprinklr_paid_historic['Total Paid Views'] = sprinklr_paid_historic['TikTok Video views (SUM)'] + sprinklr_paid_historic['Facebook Video Plays (3 sec) (SUM)']

sprinklr_paid_historic['Qualified Paid Views'] = sprinklr_paid_historic['Facebook Video Plays to 25% (SUM)'] + sprinklr_paid_historic['TikTok 6-second video views (SUM)']

sprinklr_paid_historic['Total Interactions'] = sprinklr_paid_historic['TikTok Paid comments (SUM)'] + sprinklr_paid_historic['TikTok Paid shares (SUM)'] + sprinklr_paid_historic['TikTok Paid likes (SUM)'] + sprinklr_paid_historic['Facebook Post Comments (SUM)'] + sprinklr_paid_historic['Facebook Post Shares (SUM)'] + sprinklr_paid_historic['Facebook Reactions (SUM)'] + sprinklr_paid_historic['Facebook Post Saves (SUM)']

sprinklr_paid_historic['Total Clicks'] = sprinklr_paid_historic['Facebook Link Clicks (SUM)'] + sprinklr_paid_historic['TikTok Clicks (Destination) (SUM)']

sprinklr_paid_historic["Campaign"] = (
    sprinklr_paid_historic["Paid Initiative Name"]
    .str.split("_")
    .str[5]
)

sprinklr_paid_historic["Campaign_Objective"] = (
    sprinklr_paid_historic["Paid Initiative Name"]
    .str.split("_")
    .str[8]
)

# Create month column
sprinklr_paid_historic['month_date'] = (
    pd.to_datetime(sprinklr_paid_historic['Published Date'])
    .dt.to_period('M')
    .dt.to_timestamp()
)


In [157]:

# Group values (sum) by organic_id, considering campaign and campaign objective (paid dashboard detail)
sprinklr_paid_historic_grouped_detail = sprinklr_paid_historic.groupby(['Organic_ID', 'Social Network', 'Body', 'Outbound Message Category', 'Brand (Account)', 'Country of Origin (Account)', 'Published Date', 'Campaign', 'Campaign_Objective', 'Ad Post Permalink', 'month_date'], as_index=False)[[
    'Impressions (SUM)',
    'Spent (USD) in USD (SUM)',
    'TikTok Video views (SUM)',
    'TikTok 6-second video views (SUM)',
    'Facebook Video Plays (3 sec) (SUM)',
    'Facebook Video Plays to 25% (SUM)',
    'TikTok Paid comments (SUM)',
    'TikTok Paid shares (SUM)',
    'TikTok Paid likes (SUM)',
    'Facebook Post Comments (SUM)',
    'Facebook Post Shares (SUM)',
    'Facebook Post Likes (SUM)',
    'Facebook Link Clicks (SUM)',
    'Facebook Reactions (SUM)',
    'Facebook Post Saves (SUM)',
    'Neutral Sentiment Count (Paid + Organic) (AVG)',
    'Positive Sentiment Count (Paid + Organic) (AVG)',
    'Negative Sentiment Count (Paid + Organic) (AVG)',
    'Facebook Avg. Duration of Video Played (SUM)',
    'TikTok Clicks (Destination) (SUM)',
    'Total Paid Views',
    'Qualified Paid Views',
    'Total Interactions',
    'Total Clicks'

]].sum()


In [158]:
# Save sprinklr_paid_final_table
# Open the destination sheets file
sh = gc.open_by_key('1W73RHKRuKfp-AAVQDgrMSwP3huq0r8-bDeRMLjbPxZA')
worksheet = sh.worksheet('Hoja 1')
# Replace old data with new data
set_with_dataframe(worksheet, sprinklr_paid_historic_grouped_detail)
print("DataFrame saved successfully!")

DataFrame saved successfully!


In [159]:
# Add 'Ads Count'
sprinklr_paid_historic['Ads_Count'] = 1

# Group values (sum) by month, considering campaign and campaign objective (paid dashboard overall)
sprinklr_paid_historic_grouped_month = sprinklr_paid_historic.groupby(['Social Network', 'Outbound Message Category', 'Brand (Account)', 'Country of Origin (Account)', 'Campaign', 'Campaign_Objective', 'month_date'], as_index=False)[[
    'Impressions (SUM)',
    'Spent (USD) in USD (SUM)',
    'Neutral Sentiment Count (Paid + Organic) (AVG)',
    'Positive Sentiment Count (Paid + Organic) (AVG)',
    'Negative Sentiment Count (Paid + Organic) (AVG)',
    'Total Paid Views',
    'Qualified Paid Views',
    'Total Interactions',
    'Total Clicks',
    'Ads_Count'

]].sum()

In [160]:
# Add one-month lags
# 1. Define dimensions and group keys (everything except 'month')
dimensions = [
'Social Network', 'Outbound Message Category', 'Brand (Account)', 'Country of Origin (Account)', 'Campaign', 'Campaign_Objective'
]
group_keys = [col for col in dimensions if col != 'month_date']

# 2. Identify all metric columns
metric_cols = [
    col
    for col in sprinklr_paid_historic_grouped_month.columns
    if col not in dimensions
]

# 3. Sort chronologically by month within each entity group
sprinklr_paid_historic_grouped_month = sprinklr_paid_historic_grouped_month.sort_values(
    by=group_keys + ['month_date']
)

# 4. Calculate 1-month lag per group
lagged_metrics = sprinklr_paid_historic_grouped_month.groupby(group_keys)[
    metric_cols
].shift(1)

# 5. Add the 'prev_' prefix to the lagged column names
lagged_cols = [f'prev_{col}' for col in metric_cols]
lagged_metrics.columns = lagged_cols

# 6. Append the lagged columns back to the original DataFrame
sprinklr_paid_historic_grouped_month = pd.concat(
    [sprinklr_paid_historic_grouped_month, lagged_metrics], axis=1
)

sprinklr_paid_historic_grouped_month[lagged_cols] = sprinklr_paid_historic_grouped_month[
    lagged_cols
].fillna(0)

In [161]:
# Save sprinklr_paid_historic_grouped_month (paid_dashboard)
# Open the destination sheets file
sh = gc.open_by_key('1qmc9ezuJGBTT6qfr7SF9sl1NDTzsZvaWbAkH6LFAyfc')
worksheet = sh.worksheet('Hoja 1')
# Replace old data with new data
set_with_dataframe(worksheet, sprinklr_paid_historic_grouped_month)
print("DataFrame saved successfully!")

DataFrame saved successfully!
